# 13 - Dia da Semana, Sazonalidade e Efeito Cansaço

- Qual dia da semana é mais goleiro?
- "Agosto maldito" existe? Qual mês tem mais zebras?
- Times perdem desempenho jogando com menos de 3 dias de intervalo?

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

df = pd.read_csv('../data/raw/campeonato-brasileiro-full.csv')
df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
df['ano'] = df['data'].dt.year
df['mes'] = df['data'].dt.month
df['dia_semana'] = df['data'].dt.day_name()
df['dia_semana_num'] = df['data'].dt.dayofweek  # 0=Segunda
df = df[df['ano'] >= 2003].copy()
df['total_gols'] = df['mandante_Placar'] + df['visitante_Placar']

DIAS_PT = {0: 'Segunda', 1: 'Ter\u00e7a', 2: 'Quarta', 3: 'Quinta', 4: 'Sexta', 5: 'S\u00e1bado', 6: 'Domingo'}
df['dia_pt'] = df['dia_semana_num'].map(DIAS_PT)

print(f'Partidas: {len(df)}')
print(f'\nJogos por dia da semana:')
print(df['dia_pt'].value_counts().to_string())

Partidas: 9225

Jogos por dia da semana:
dia_pt
Domingo    4055
Sábado     2378
Quarta     1702
Quinta      711
Segunda     213
Terça       122
Sexta        44


In [2]:
# GOLS POR DIA DA SEMANA
gols_dia = df.groupby('dia_semana_num').agg(
    dia=('dia_pt', 'first'),
    media_gols=('total_gols', 'mean'),
    jogos=('ID', 'count'),
    total_gols=('total_gols', 'sum')
).reset_index()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=gols_dia['dia'],
    y=gols_dia['media_gols'],
    marker_color=['#e74c3c' if g == gols_dia['media_gols'].max() else '#3498db' 
                  for g in gols_dia['media_gols']],
    text=gols_dia['media_gols'].round(2),
    textposition='outside',
    hovertemplate='%{x}<br>Média: %{y:.2f} gols/jogo<br>Jogos: %{customdata}<extra></extra>',
    customdata=gols_dia['jogos']
))

fig.add_hline(y=df['total_gols'].mean(), line_dash='dash', line_color='gray',
              annotation_text=f"Média geral: {df['total_gols'].mean():.2f}")

fig.update_layout(
    title='Média de Gols por Dia da Semana<br><sup>Jogos de quarta à noite são mais abertos?</sup>',
    xaxis_title='Dia da Semana',
    yaxis_title='Média de Gols/Partida',
    template='plotly_white',
    width=800,
    height=450
)

fig.show()
path = '../charts/gols_dia_semana.html'
fig.write_html(path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Comparação da média de gols por dia da semana: a dispersão entre as categorias é baixa, com quinta-feira apresentando o maior valor e segunda o menor."
with open(path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/gols_dia_semana.html')

Gráfico exportado: charts/gols_dia_semana.html


In [3]:
# AGOSTO MALDITO
# Definir 'zebra': mandante perde (já que mandante é favorito ~50% das vezes)
# Ou podemos calcular por 'vitória do visitante' que é resultado menos esperado
MESES_PT = {1: 'Jan', 2: 'Fev', 3: 'Mar', 4: 'Abr', 5: 'Mai', 6: 'Jun',
            7: 'Jul', 8: 'Ago', 9: 'Set', 10: 'Out', 11: 'Nov', 12: 'Dez'}
df['mes_pt'] = df['mes'].map(MESES_PT)

df['zebra'] = df['mandante_Placar'] < df['visitante_Placar']  # Visitante vence = zebra

zebra_mes = df.groupby('mes').agg(
    pct_zebra=('zebra', 'mean'),
    jogos=('ID', 'count'),
    media_gols=('total_gols', 'mean')
).reset_index()
zebra_mes['mes_pt'] = zebra_mes['mes'].map(MESES_PT)
zebra_mes['pct_zebra'] *= 100

fig2 = make_subplots(rows=2, cols=1, 
                     subplot_titles=['% de Zebras (vitória visitante) por Mês', 
                                     'Média de Gols por Mês'],
                     vertical_spacing=0.15)

fig2.add_trace(go.Bar(
    x=zebra_mes['mes_pt'],
    y=zebra_mes['pct_zebra'],
    marker_color=['#e74c3c' if m == 8 else '#95a5a6' for m in zebra_mes['mes']],
    text=zebra_mes['pct_zebra'].round(1).astype(str) + '%',
    textposition='outside',
    hovertemplate='%{x}: %{y:.1f}% zebras<extra></extra>'
), row=1, col=1)

fig2.add_trace(go.Bar(
    x=zebra_mes['mes_pt'],
    y=zebra_mes['media_gols'],
    marker_color=['#2ecc71' if g == zebra_mes['media_gols'].max() else '#3498db'
                  for g in zebra_mes['media_gols']],
    text=zebra_mes['media_gols'].round(2),
    textposition='outside',
    hovertemplate='%{x}: %{y:.2f} gols/jogo<extra></extra>'
), row=2, col=1)

fig2.update_layout(
    title='Agosto Maldito? Sazonalidade no Brasileirão<br><sup>Agosto em vermelho</sup>',
    template='plotly_white',
    width=900,
    height=650,
    showlegend=False
)

fig2.show()
path = '../charts/sazonalidade.html'
fig2.write_html(path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Análise sazonal da taxa de zebras e média de gols por mês, testando a hipótese do 'agosto maldito'. Março apresenta o menor percentual de zebras (outlier inferior)."
with open(path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/sazonalidade.html')

Gráfico exportado: charts/sazonalidade.html


In [4]:
# EFEITO CANSAÇO
# Calcular intervalo entre jogos para cada time
df_sorted = df.sort_values('data')

jogos_time = []
for _, row in df_sorted.iterrows():
    jogos_time.append({'time': row['mandante'], 'data': row['data'], 'ano': row['ano'],
                       'gols_pro': row['mandante_Placar'], 'gols_contra': row['visitante_Placar'],
                       'local': 'casa'})
    jogos_time.append({'time': row['visitante'], 'data': row['data'], 'ano': row['ano'],
                       'gols_pro': row['visitante_Placar'], 'gols_contra': row['mandante_Placar'],
                       'local': 'fora'})

df_jogos = pd.DataFrame(jogos_time)
df_jogos = df_jogos.sort_values(['time', 'data'])
df_jogos['dias_descanso'] = df_jogos.groupby(['time', 'ano'])['data'].diff().dt.days

# Resultado
def resultado(row):
    if pd.isna(row['gols_pro']) or pd.isna(row['gols_contra']): return None
    if row['gols_pro'] > row['gols_contra']: return 3
    elif row['gols_pro'] == row['gols_contra']: return 1
    else: return 0

df_jogos['pontos'] = df_jogos.apply(resultado, axis=1)
df_jogos = df_jogos.dropna(subset=['dias_descanso', 'pontos'])

# Agrupar por faixa de descanso
def faixa_descanso(d):
    if d <= 2: return '\u2264 2 dias'
    elif d <= 4: return '3-4 dias'
    elif d <= 7: return '5-7 dias'
    else: return '8+ dias'

df_jogos['faixa_desc'] = df_jogos['dias_descanso'].apply(faixa_descanso)
df_jogos['faixa_desc'] = pd.Categorical(df_jogos['faixa_desc'], 
                                         ['\u2264 2 dias', '3-4 dias', '5-7 dias', '8+ dias'])

aprov = df_jogos.groupby('faixa_desc').agg(
    aproveitamento=('pontos', lambda x: x.mean() / 3 * 100),
    jogos=('pontos', 'count')
).reset_index()

print('Aproveitamento por intervalo de descanso:')
for _, r in aprov.iterrows():
    print(f"  {r['faixa_desc']}: {r['aproveitamento']:.1f}% ({int(r['jogos'])} jogos)")

fig3 = px.bar(aprov, x='faixa_desc', y='aproveitamento',
              text=aprov['aproveitamento'].round(1).astype(str) + '%',
              title='Efeito Cansaço: Aproveitamento por Intervalo de Descanso<br><sup>Times jogam pior com menos dias de intervalo?</sup>',
              labels={'faixa_desc': 'Dias de Descanso', 'aproveitamento': 'Aproveitamento (%)'},
              color='aproveitamento',
              color_continuous_scale='RdYlGn')
fig3.update_layout(template='plotly_white', width=700, height=450, showlegend=False)
fig3.update_traces(textposition='outside')

fig3.show()
path = '../charts/efeito_cansaco.html'
fig3.write_html(path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Análise comparativa do aproveitamento médio por faixa de intervalo de descanso. A diferença entre as categorias é reduzida (~3.6 p.p.), sugerindo baixa significância estatística do efeito cansaço sobre o resultado."
with open(path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/efeito_cansaco.html')

Aproveitamento por intervalo de descanso:
  ≤ 2 dias: 42.4% (77 jogos)
  3-4 dias: 46.0% (8107 jogos)
  5-7 dias: 45.5% (6970 jogos)
  8+ dias: 45.1% (2802 jogos)


Gráfico exportado: charts/efeito_cansaco.html
